# 05 - Build `ml.trip_validity_dataset`

The final, ML-ready table for the Trip Validity model: every identifier,
every raw metric from `ml.trip_validity_trip_metrics`, and every
relative/normalized column surveyed and agreed on beforehand - 80
columns total (15 identifiers + 40 existing metrics/counts + 25 new
relative columns).

Built with a single `CREATE TABLE ... AS SELECT` from
`ml.trip_validity_trip_metrics` joined to `ml.trip_validity_trips` (for
`bus_id`/`trip_date`/`trip_hour`/`trip_opening_timestamp`/
`trip_closing_timestamp`, which live on `trips` but weren't carried onto
`trip_metrics`) and left-joined to `ml.trip_validity_bus_avl_match`
(built in `04_avl_positions.ipynb`, for `avl_matched`/`avl_match_source`)
- no expensive spatial recomputation, just arithmetic on already-materialized
columns plus two cheap key lookups, so this should run in seconds, not
minutes.

**Column order**: identifiers first, then five themed groups (duration,
fares, geometry, path match, directional progress), each ordered raw →
count → relative, so a raw value and the ratio computed from it always
sit next to each other.

**Two conventions applied uniformly**:
- Every ratio divides through `NULLIF(denominator, 0)`, since Postgres's
  floating-point division does not error on divide-by-zero - it silently
  returns `Infinity`, which would sit in the table looking like a real
  number and quietly break anything downstream.
- The four `path_match_score_*` columns are clipped to `[0, 1]` via
  `GREATEST(0, ...)`- but `GREATEST`/`LEAST` in Postgres **ignore NULL
  arguments** rather than propagating them (unlike plain arithmetic,
  which is NULL-safe), so `GREATEST(0, NULL)` silently returns `0`, not
  `NULL`. Left unguarded, a trip with no matched shape would wrongly
  show a match score of `0` ("worst possible match") instead of `NULL`
  ("not applicable"). Each of those four columns is wrapped in a `CASE`
  that checks whether the *unclipped* ratio is `NULL` first, and only
  applies `GREATEST` when it's a real number.

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
print("connected")

connected


## Build the table

In [4]:
conn.execute("DROP TABLE IF EXISTS ml.trip_validity_dataset CASCADE;")

conn.execute("""
    CREATE TABLE ml.trip_validity_dataset AS
    SELECT
        -- ===== Identifiers =====
        tm.trip_id,
        t.bus_id,
        tm.route_id,
        tm.route_direction,
        t.trip_date,
        t.trip_hour,
        t.trip_opening_timestamp,
        t.trip_closing_timestamp,
        tm.gtfs_feed_version_date,
        tm.gtfs_route_short_name,
        tm.gtfs_shape_id_i,
        tm.gtfs_shape_id_v,
        tm.gtfs_route_has_both_directions,

        -- ===== AVL position matching =====
        COALESCE(bm.avl_matched, false) AS avl_matched,
        bm.avl_match_source,

        -- ===== Duration vs. expectations =====
        tm.trip_duration_seconds,

        tm.route_avg_trip_duration_seconds_loo,
        tm.route_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_avg,

        tm.route_direction_avg_trip_duration_seconds_loo,
        tm.route_direction_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_direction_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_direction_avg,

        tm.route_hour_avg_trip_duration_seconds_loo,
        tm.route_hour_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_hour_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_hour_avg,

        tm.route_direction_hour_avg_trip_duration_seconds_loo,
        tm.route_direction_hour_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_direction_hour_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_direction_hour_avg,

        tm.route_reverse_direction_avg_trip_duration_seconds,
        tm.route_reverse_direction_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_reverse_direction_avg_trip_duration_seconds, 0)
            AS trip_duration_ratio_to_reverse_direction_avg,

        tm.route_reverse_direction_hour_avg_trip_duration_seconds,
        tm.route_reverse_direction_hour_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_reverse_direction_hour_avg_trip_duration_seconds, 0)
            AS trip_duration_ratio_to_reverse_direction_hour_avg,

        tm.route_i_scheduled_duration_avg_seconds,
        tm.route_i_scheduled_duration_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_i_scheduled_duration_avg_seconds, 0)
            AS trip_duration_ratio_to_scheduled_i,

        tm.route_v_scheduled_duration_avg_seconds,
        tm.route_v_scheduled_duration_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_v_scheduled_duration_avg_seconds, 0)
            AS trip_duration_ratio_to_scheduled_v,

        tm.route_i_scheduled_duration_avg_seconds_at_hour,
        tm.route_i_scheduled_duration_n_trips_at_hour,
        tm.trip_duration_seconds
            / NULLIF(tm.route_i_scheduled_duration_avg_seconds_at_hour, 0)
            AS trip_duration_ratio_to_scheduled_i_at_hour,

        tm.route_v_scheduled_duration_avg_seconds_at_hour,
        tm.route_v_scheduled_duration_n_trips_at_hour,
        tm.trip_duration_seconds
            / NULLIF(tm.route_v_scheduled_duration_avg_seconds_at_hour, 0)
            AS trip_duration_ratio_to_scheduled_v_at_hour,

        -- ===== Fare activity =====
        tm.trip_fare_count,
        tm.fare_gap_avg_seconds,
        tm.fare_gap_stddev_seconds,
        tm.fare_gap_stddev_seconds / NULLIF(tm.fare_gap_avg_seconds, 0)
            AS fare_gap_coefficient_of_variation,
        tm.fare_gap_avg_seconds / NULLIF(tm.trip_duration_seconds, 0)
            AS fare_gap_avg_ratio_to_duration,
        tm.fare_span_seconds,
        tm.fare_span_seconds / NULLIF(tm.trip_duration_seconds, 0)
            AS fare_span_ratio_to_duration,

        -- ===== Trip geometry vs. route length =====
        tm.trip_distance_meters,
        tm.route_i_length_meters,
        tm.trip_distance_meters / NULLIF(tm.route_i_length_meters, 0)
            AS trip_distance_ratio_to_route_i,
        tm.route_v_length_meters,
        tm.trip_distance_meters / NULLIF(tm.route_v_length_meters, 0)
            AS trip_distance_ratio_to_route_v,
        tm.trip_points_standard_distance_meters,
        tm.trip_points_standard_distance_meters
            / NULLIF(tm.route_i_length_meters, 0)
            AS trip_cohesion_ratio_to_route_i,
        tm.trip_points_standard_distance_meters
            / NULLIF(tm.route_v_length_meters, 0)
            AS trip_cohesion_ratio_to_route_v,

        -- ===== Path shape match =====
        tm.trip_start_distance_to_i_start_meters,
        tm.trip_start_distance_to_i_start_meters
            / NULLIF(tm.route_i_length_meters, 0)
            AS trip_start_offset_ratio_to_i,
        tm.trip_start_distance_to_v_start_meters,
        tm.trip_start_distance_to_v_start_meters
            / NULLIF(tm.route_v_length_meters, 0)
            AS trip_start_offset_ratio_to_v,
        tm.trip_end_distance_to_i_end_meters,
        tm.trip_end_distance_to_i_end_meters
            / NULLIF(tm.route_i_length_meters, 0)
            AS trip_end_offset_ratio_to_i,
        tm.trip_end_distance_to_v_end_meters,
        tm.trip_end_distance_to_v_end_meters
            / NULLIF(tm.route_v_length_meters, 0)
            AS trip_end_offset_ratio_to_v,

        tm.path_frechet_distance_to_i_meters,
        CASE WHEN 1 - tm.path_frechet_distance_to_i_meters
                      / NULLIF(tm.route_i_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_frechet_distance_to_i_meters
                                  / NULLIF(tm.route_i_length_meters, 0))
        END AS path_match_score_frechet_i,

        tm.path_frechet_distance_to_v_meters,
        CASE WHEN 1 - tm.path_frechet_distance_to_v_meters
                      / NULLIF(tm.route_v_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_frechet_distance_to_v_meters
                                  / NULLIF(tm.route_v_length_meters, 0))
        END AS path_match_score_frechet_v,

        tm.path_hausdorff_distance_to_i_meters,
        CASE WHEN 1 - tm.path_hausdorff_distance_to_i_meters
                      / NULLIF(tm.route_i_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_hausdorff_distance_to_i_meters
                                  / NULLIF(tm.route_i_length_meters, 0))
        END AS path_match_score_hausdorff_i,

        tm.path_hausdorff_distance_to_v_meters,
        CASE WHEN 1 - tm.path_hausdorff_distance_to_v_meters
                      / NULLIF(tm.route_v_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_hausdorff_distance_to_v_meters
                                  / NULLIF(tm.route_v_length_meters, 0))
        END AS path_match_score_hausdorff_v,

        -- ===== Directional progress =====
        tm.trip_progress_correlation_to_i,
        tm.trip_progress_correlation_to_v,
        tm.trip_progress_correlation_n_points

    FROM ml.trip_validity_trip_metrics tm
    JOIN ml.trip_validity_trips t ON t.trip_id = tm.trip_id
    LEFT JOIN ml.trip_validity_bus_avl_match bm ON bm.bus_id = t.bus_id
    ORDER BY tm.trip_id;
""")
conn.commit()

EXPECTED_TRIP_COUNT = 940988
EXPECTED_COLUMN_COUNT = 80

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_dataset;")
    row_count = cur.fetchone()[0]
    print("rows:", row_count)
    cur.execute("""
        SELECT count(*) FROM information_schema.columns
        WHERE table_schema = 'ml' AND table_name = 'trip_validity_dataset';
    """)
    column_count = cur.fetchone()[0]
    print("columns:", column_count)

if row_count != EXPECTED_TRIP_COUNT:
    msg = "row count mismatch against trip_validity_trips"
    raise AssertionError(msg)
if column_count != EXPECTED_COLUMN_COUNT:
    msg = "column count mismatch against the plan"
    raise AssertionError(msg)

rows: 940988
columns: 80


## Constraints and indexes

`CREATE TABLE ... AS SELECT` doesn't carry over constraints, so adding
them explicitly: the primary key (and FK back to `trip_validity_trips`,
for traceability), `NOT NULL` on the columns known to always be
populated (mirroring their source tables' own guarantees), and indexes
on the columns most likely to be used for filtering/sampling during
active learning.

In [5]:
conn.execute("""
    ALTER TABLE ml.trip_validity_dataset
        ADD PRIMARY KEY (trip_id),
        ADD FOREIGN KEY (trip_id) REFERENCES ml.trip_validity_trips (trip_id),
        ALTER COLUMN bus_id SET NOT NULL,
        ALTER COLUMN route_id SET NOT NULL,
        ALTER COLUMN route_direction SET NOT NULL,
        ALTER COLUMN trip_date SET NOT NULL,
        ALTER COLUMN trip_hour SET NOT NULL,
        ALTER COLUMN trip_opening_timestamp SET NOT NULL,
        ALTER COLUMN trip_closing_timestamp SET NOT NULL,
        ALTER COLUMN avl_matched SET NOT NULL,
        ALTER COLUMN trip_fare_count SET NOT NULL;
""")

conn.execute("""
    CREATE INDEX trip_validity_dataset_route_id_idx
        ON ml.trip_validity_dataset (route_id);
    CREATE INDEX trip_validity_dataset_route_direction_idx
        ON ml.trip_validity_dataset (route_id, route_direction);
    CREATE INDEX trip_validity_dataset_bus_id_idx
        ON ml.trip_validity_dataset (bus_id);
    CREATE INDEX trip_validity_dataset_trip_date_idx
        ON ml.trip_validity_dataset (trip_date);
    ANALYZE ml.trip_validity_dataset;
""")
conn.commit()
print("constraints and indexes applied")

constraints and indexes applied


## Column-level provenance comments

In [6]:
def comment_on_column(cur: psycopg.Cursor, table: str, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one column via safe SQL composition."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.{}.{} IS {};").format(
            sql.Identifier(table), sql.Identifier(col), sql.Literal(text)
        )
    )


NULLIF_NOTE = (
    " Ratio divides through NULLIF(denominator, 0), so a zero denominator "
    "gives NULL rather than the Infinity Postgres would otherwise "
    "silently produce."
)
CLIP_NOTE = (
    " Clipped to [0, 1] via GREATEST(0, ...), NULL-safely (unlike bare "
    "GREATEST, which would turn a NULL input into 0)."
)

DATASET_COMMENTS = {
    # Identifiers
    "trip_id": (
        "PK. = ml.trip_validity_trips.trip_id / ml.trip_validity_trip_metrics.trip_id."
    ),
    "bus_id": "From ml.trip_validity_trips.bus_id.",
    "route_id": "From ml.trip_validity_trip_metrics.route_id.",
    "route_direction": (
        "From ml.trip_validity_trip_metrics.route_direction (this trip's "
        "own observed AFC direction)."
    ),
    "trip_date": "From ml.trip_validity_trips.trip_date.",
    "trip_hour": "From ml.trip_validity_trips.trip_hour.",
    "trip_opening_timestamp": (
        "From ml.trip_validity_trips.trip_opening_timestamp (UTC). Start "
        "of the AVL position window used to build "
        "ml.trip_validity_trip_positions."
    ),
    "trip_closing_timestamp": (
        "From ml.trip_validity_trips.trip_closing_timestamp (UTC). End of "
        "the AVL position window. May be the 1899-12-30 Delphi zero-date "
        "sentinel (see trip_duration_seconds) - such trips have no "
        "positions, not garbage ones, since closing < opening yields an "
        "empty window."
    ),
    "gtfs_feed_version_date": (
        "From ml.trip_validity_trip_metrics (originally "
        "ml.trip_validity_route_gtfs_match)."
    ),
    "gtfs_route_short_name": (
        "From ml.trip_validity_trip_metrics - the GTFS join key actually used."
    ),
    "gtfs_shape_id_i": "From ml.trip_validity_trip_metrics.",
    "gtfs_shape_id_v": "From ml.trip_validity_trip_metrics.",
    "gtfs_route_has_both_directions": (
        "From ml.trip_validity_trip_metrics. NULL = route never matched "
        "any GTFS feed; false = matched but only one direction has a "
        "shape; true = both directions have a shape."
    ),
    "avl_matched": (
        "From ml.trip_validity_bus_avl_match, keyed by bus_id (not "
        "trip_id - every trip on the same bus shares the same match). "
        "True iff this trip's bus_id was resolved to an AVL vehicle_id "
        "or device_id, i.e. ml.trip_validity_trip_positions can have rows "
        "for this trip. False (never NULL) otherwise."
    ),
    "avl_match_source": (
        "Which silver crosswalk resolved the match: "
        "'dictionary_device' (silver.dictionary_device.codigo -> "
        "device_id, preferred when both match) or 'dictionary_vehicle' "
        "(silver.dictionary_vehicle.cod_veiculo -> id_veiculo, used only "
        "when no dictionary_device match exists). NULL iff avl_matched "
        "is false."
    ),
    # Duration
    "trip_duration_seconds": (
        "From ml.trip_validity_trip_metrics. NULL for the 9 trips with "
        "the 1899-12-30 Delphi zero-date sentinel closing time."
    ),
    "route_avg_trip_duration_seconds_loo": (
        "Leave-one-out mean duration across other trips on this route."
    ),
    "route_avg_trip_duration_n_trips_loo": "Count backing the average above.",
    "trip_duration_ratio_to_route_avg": (
        "trip_duration_seconds / route_avg_trip_duration_seconds_loo. "
        "1.0 = exactly average; >1 = slower; <1 = faster." + NULLIF_NOTE
    ),
    "route_direction_avg_trip_duration_seconds_loo": (
        "Same as route_avg_trip_duration_seconds_loo, grouped by "
        "(route_id, route_direction)."
    ),
    "route_direction_avg_trip_duration_n_trips_loo": (
        "Count backing the average above."
    ),
    "trip_duration_ratio_to_route_direction_avg": (
        "trip_duration_seconds / "
        "route_direction_avg_trip_duration_seconds_loo." + NULLIF_NOTE
    ),
    "route_hour_avg_trip_duration_seconds_loo": (
        "Same, grouped by (route_id, trip_hour)."
    ),
    "route_hour_avg_trip_duration_n_trips_loo": "Count backing the average above.",
    "trip_duration_ratio_to_route_hour_avg": (
        "trip_duration_seconds / route_hour_avg_trip_duration_seconds_loo."
        + NULLIF_NOTE
    ),
    "route_direction_hour_avg_trip_duration_seconds_loo": (
        "Same, grouped by (route_id, route_direction, trip_hour)."
    ),
    "route_direction_hour_avg_trip_duration_n_trips_loo": (
        "Count backing the average above."
    ),
    "trip_duration_ratio_to_route_direction_hour_avg": (
        "trip_duration_seconds / "
        "route_direction_hour_avg_trip_duration_seconds_loo." + NULLIF_NOTE
    ),
    "route_reverse_direction_avg_trip_duration_seconds": (
        "Mean duration of trips on this route going the OPPOSITE "
        "route_direction. Not leave-one-out (this trip can't be a "
        "member of that group)."
    ),
    "route_reverse_direction_n_trips": "Count backing the average above.",
    "trip_duration_ratio_to_reverse_direction_avg": (
        "trip_duration_seconds / "
        "route_reverse_direction_avg_trip_duration_seconds. How this "
        "trip compares to the OTHER direction's typical duration." + NULLIF_NOTE
    ),
    "route_reverse_direction_hour_avg_trip_duration_seconds": (
        "Same as route_reverse_direction_avg_trip_duration_seconds, "
        "restricted to the opposite direction's trips at this row's "
        "trip_hour."
    ),
    "route_reverse_direction_hour_n_trips": "Count backing the average above.",
    "trip_duration_ratio_to_reverse_direction_hour_avg": (
        "trip_duration_seconds / "
        "route_reverse_direction_hour_avg_trip_duration_seconds." + NULLIF_NOTE
    ),
    "route_i_scheduled_duration_avg_seconds": (
        "GTFS-scheduled average duration for direction I on this "
        "route/feed (from silver.gtfs_stop_times, via "
        "ml.trip_validity_route_schedule). What the TIMETABLE says, not "
        "other real trips."
    ),
    "route_i_scheduled_duration_n_trips": (
        "Count of scheduled GTFS trips backing the average above."
    ),
    "trip_duration_ratio_to_scheduled_i": (
        "trip_duration_seconds / route_i_scheduled_duration_avg_seconds. "
        "Actual vs. planned." + NULLIF_NOTE
    ),
    "route_v_scheduled_duration_avg_seconds": (
        "Same as route_i_scheduled_duration_avg_seconds, direction V."
    ),
    "route_v_scheduled_duration_n_trips": "Count backing the average above.",
    "trip_duration_ratio_to_scheduled_v": (
        "trip_duration_seconds / route_v_scheduled_duration_avg_seconds." + NULLIF_NOTE
    ),
    "route_i_scheduled_duration_avg_seconds_at_hour": (
        "Same as route_i_scheduled_duration_avg_seconds, restricted to "
        "scheduled GTFS trips whose start_hour matches this row's "
        "trip_hour."
    ),
    "route_i_scheduled_duration_n_trips_at_hour": ("Count backing the average above."),
    "trip_duration_ratio_to_scheduled_i_at_hour": (
        "trip_duration_seconds / "
        "route_i_scheduled_duration_avg_seconds_at_hour. Actual vs. "
        "planned, hour-matched." + NULLIF_NOTE
    ),
    "route_v_scheduled_duration_avg_seconds_at_hour": (
        "Same as route_i_scheduled_duration_avg_seconds_at_hour, direction V."
    ),
    "route_v_scheduled_duration_n_trips_at_hour": ("Count backing the average above."),
    "trip_duration_ratio_to_scheduled_v_at_hour": (
        "trip_duration_seconds / "
        "route_v_scheduled_duration_avg_seconds_at_hour." + NULLIF_NOTE
    ),
    # Fares
    "trip_fare_count": (
        "From ml.trip_validity_trip_metrics. count(*) of all fare taps "
        "on this trip, geo-tagged or not."
    ),
    "fare_gap_avg_seconds": (
        "Mean gap between consecutive fare taps, ordered by boarding_at. "
        "NULL if only 1 fare."
    ),
    "fare_gap_stddev_seconds": (
        "Sample stddev of the same gaps. NULL if fewer than 2 gaps exist."
    ),
    "fare_gap_coefficient_of_variation": (
        "fare_gap_stddev_seconds / fare_gap_avg_seconds. Standard "
        "normalized-dispersion measure: were the gaps steady or erratic, "
        "independent of trip length." + NULLIF_NOTE
    ),
    "fare_gap_avg_ratio_to_duration": (
        "fare_gap_avg_seconds / trip_duration_seconds. What fraction of "
        "the whole trip is 'typical time between taps'." + NULLIF_NOTE
    ),
    "fare_span_seconds": (
        "max(boarding_at) - min(boarding_at) across all fares. 0 (not "
        "NULL) for a 1-fare trip."
    ),
    "fare_span_ratio_to_duration": (
        "fare_span_seconds / trip_duration_seconds. What fraction of "
        "the trip's duration actually had fare activity." + NULLIF_NOTE
    ),
    # Geometry
    "trip_distance_meters": (
        "ST_Length(trip_path::geography). NULL if trip_path is NULL "
        "(fewer than 2 geo-tagged fares)."
    ),
    "route_i_length_meters": (
        "Official route length, direction I (ml.trip_validity_route_shapes)."
    ),
    "trip_distance_ratio_to_route_i": (
        "trip_distance_meters / route_i_length_meters. Did the GPS trace "
        "cover roughly the whole route, or just a fraction of it?" + NULLIF_NOTE
    ),
    "route_v_length_meters": "Official route length, direction V.",
    "trip_distance_ratio_to_route_v": (
        "trip_distance_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    "trip_points_standard_distance_meters": (
        "RMS distance of this trip's geo-tagged fares from their own "
        "centroid (spatial-statistics standard distance). 0 for a "
        "single point, NULL for zero."
    ),
    "trip_cohesion_ratio_to_route_i": (
        "trip_points_standard_distance_meters / route_i_length_meters. "
        "Is the point spread large or small relative to how long this "
        "route actually is?" + NULLIF_NOTE
    ),
    "trip_cohesion_ratio_to_route_v": (
        "trip_points_standard_distance_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    # Path match
    "trip_start_distance_to_i_start_meters": (
        "ST_Distance (geography) between the trip's first GPS point and "
        "the official I-direction route's start point."
    ),
    "trip_start_offset_ratio_to_i": (
        "trip_start_distance_to_i_start_meters / route_i_length_meters." + NULLIF_NOTE
    ),
    "trip_start_distance_to_v_start_meters": (
        "Same as trip_start_distance_to_i_start_meters, direction V."
    ),
    "trip_start_offset_ratio_to_v": (
        "trip_start_distance_to_v_start_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    "trip_end_distance_to_i_end_meters": (
        "ST_Distance (geography) between the trip's last GPS point and "
        "the official I-direction route's end point."
    ),
    "trip_end_offset_ratio_to_i": (
        "trip_end_distance_to_i_end_meters / route_i_length_meters." + NULLIF_NOTE
    ),
    "trip_end_distance_to_v_end_meters": (
        "Same as trip_end_distance_to_i_end_meters, direction V."
    ),
    "trip_end_offset_ratio_to_v": (
        "trip_end_distance_to_v_end_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    "path_frechet_distance_to_i_meters": (
        "ST_FrechetDistance between the trip's path and the I-direction "
        "shape (both projected to EPSG:31984 first). Order-aware path "
        "similarity distance, meters."
    ),
    "path_match_score_frechet_i": (
        "1 - (path_frechet_distance_to_i_meters / route_i_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    "path_frechet_distance_to_v_meters": (
        "Same as path_frechet_distance_to_i_meters, direction V."
    ),
    "path_match_score_frechet_v": (
        "1 - (path_frechet_distance_to_v_meters / route_v_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    "path_hausdorff_distance_to_i_meters": (
        "ST_HausdorffDistance between the trip's path and the "
        "I-direction shape (both projected to EPSG:31984 first). "
        "Order-agnostic 'largest gap' path similarity distance, meters."
    ),
    "path_match_score_hausdorff_i": (
        "1 - (path_hausdorff_distance_to_i_meters / route_i_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    "path_hausdorff_distance_to_v_meters": (
        "Same as path_hausdorff_distance_to_i_meters, direction V."
    ),
    "path_match_score_hausdorff_v": (
        "1 - (path_hausdorff_distance_to_v_meters / route_v_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    # Directional progress
    "trip_progress_correlation_to_i": (
        "Pearson correlation between position-along-the-I-shape "
        "(ST_LineLocatePoint) and boarding_at, across this trip's "
        "geo-tagged fares. +1 = steady forward progress, -1 = steady "
        "reverse progress, ~0 = no consistent relationship. Already "
        "bounded [-1, 1] by construction - not further normalized."
    ),
    "trip_progress_correlation_to_v": (
        "Same as trip_progress_correlation_to_i, direction V."
    ),
    "trip_progress_correlation_n_points": (
        "Count of geo-tagged fares used for both correlations above. A "
        "value of 2 means the correlation is a meaningless exact +-1 (a "
        "line through 2 points is always 'perfectly correlated') - use "
        "this to judge how much to trust the two correlation columns."
    ),
}

with conn.cursor() as cur:
    for col, text in DATASET_COMMENTS.items():
        comment_on_column(cur, "trip_validity_dataset", col, text)

conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_dataset IS {};").format(
        sql.Literal(
            "Trip Validity model: the final ML-ready dataset. One row per "
            "trip, 80 columns (identifiers + raw metrics from "
            "ml.trip_validity_trip_metrics + every relative/normalized "
            "column). See ml/trip_validity_model/notebooks/"
            "05_final_dataset.ipynb."
        )
    )
)
conn.commit()
print(f"comments applied for {len(DATASET_COMMENTS)} columns")

comments applied for 80 columns


## Verification

Row/column counts, a check that no ratio silently produced `Infinity`
(would mean a `NULLIF` guard was missed somewhere), and a hand-check of
one duration ratio against its raw inputs.

In [7]:
RATIO_COLUMNS = [
    "trip_duration_ratio_to_route_avg",
    "trip_duration_ratio_to_route_direction_avg",
    "trip_duration_ratio_to_route_hour_avg",
    "trip_duration_ratio_to_route_direction_hour_avg",
    "trip_duration_ratio_to_reverse_direction_avg",
    "trip_duration_ratio_to_reverse_direction_hour_avg",
    "trip_duration_ratio_to_scheduled_i",
    "trip_duration_ratio_to_scheduled_v",
    "trip_duration_ratio_to_scheduled_i_at_hour",
    "trip_duration_ratio_to_scheduled_v_at_hour",
    "fare_gap_coefficient_of_variation",
    "fare_gap_avg_ratio_to_duration",
    "fare_span_ratio_to_duration",
    "trip_distance_ratio_to_route_i",
    "trip_distance_ratio_to_route_v",
    "trip_cohesion_ratio_to_route_i",
    "trip_cohesion_ratio_to_route_v",
    "trip_start_offset_ratio_to_i",
    "trip_start_offset_ratio_to_v",
    "trip_end_offset_ratio_to_i",
    "trip_end_offset_ratio_to_v",
    "path_match_score_frechet_i",
    "path_match_score_frechet_v",
    "path_match_score_hausdorff_i",
    "path_match_score_hausdorff_v",
]
EXPECTED_RATIO_COLUMN_COUNT = 25
FLOAT_TOLERANCE = 1e-9

if len(RATIO_COLUMNS) != EXPECTED_RATIO_COLUMN_COUNT:
    msg = (
        f"expected {EXPECTED_RATIO_COLUMN_COUNT} new ratio columns, "
        f"listed {len(RATIO_COLUMNS)}"
    )
    raise AssertionError(msg)

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_dataset;")
    print("rows:", cur.fetchone()[0])

    # no silent Infinity anywhere - built via safe sql.Identifier
    # composition, not raw string interpolation, even though
    # RATIO_COLUMNS is a fixed, hardcoded list (not user input)
    infinity_check = sql.SQL(" + ").join(
        sql.SQL(
            "count(*) FILTER (WHERE {0} = 'Infinity'::double precision "
            "OR {0} = '-Infinity'::double precision)"
        ).format(sql.Identifier(c))
        for c in RATIO_COLUMNS
    )
    cur.execute(
        sql.SQL("SELECT {} FROM ml.trip_validity_dataset;").format(infinity_check)
    )
    n_infinite = cur.fetchone()[0]
    print("rows with an Infinity value across all 25 ratio columns:", n_infinite)
    if n_infinite != 0:
        msg = "found Infinity - a NULLIF guard was missed"
        raise AssertionError(msg)

    # path_match_score stays in [0, 1] wherever it's not NULL
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_dataset
        WHERE (path_match_score_frechet_i IS NOT NULL
               AND (path_match_score_frechet_i < 0
                    OR path_match_score_frechet_i > 1))
           OR (path_match_score_frechet_v IS NOT NULL
               AND (path_match_score_frechet_v < 0
                    OR path_match_score_frechet_v > 1))
           OR (path_match_score_hausdorff_i IS NOT NULL
               AND (path_match_score_hausdorff_i < 0
                    OR path_match_score_hausdorff_i > 1))
           OR (path_match_score_hausdorff_v IS NOT NULL
               AND (path_match_score_hausdorff_v < 0
                    OR path_match_score_hausdorff_v > 1));
    """)
    out_of_range = cur.fetchone()[0]
    print("path_match_score rows out of [0,1]:", out_of_range)
    if out_of_range != 0:
        msg = "path_match_score found out of [0,1] range"
        raise AssertionError(msg)

    # NULL-safety check: path_match_score must be NULL exactly when
    # frechet_i is NULL, never a stray 0
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_dataset
        WHERE path_frechet_distance_to_i_meters IS NULL
          AND path_match_score_frechet_i IS NOT NULL;
    """)
    leaked_zero = cur.fetchone()[0]
    print(
        "rows where a NULL frechet distance wrongly produced a non-null score:",
        leaked_zero,
    )
    if leaked_zero != 0:
        msg = "a NULL frechet distance wrongly produced a non-null score"
        raise AssertionError(msg)

    # hand-check one ratio against its raw inputs, on a real row
    cur.execute("""
        SELECT trip_id, trip_duration_seconds,
               route_avg_trip_duration_seconds_loo,
               trip_duration_ratio_to_route_avg
        FROM ml.trip_validity_dataset
        WHERE trip_duration_ratio_to_route_avg IS NOT NULL
        ORDER BY trip_id LIMIT 1;
    """)
    trip_id, dur, avg, ratio = cur.fetchone()
    print(
        f"hand-check trip {trip_id}: {dur} / {avg} = {dur / avg:.6f}  "
        f"(stored: {ratio:.6f})"
    )
    if abs(dur / avg - ratio) >= FLOAT_TOLERANCE:
        msg = "hand-check ratio does not match stored value"
        raise AssertionError(msg)

print("all checks passed")

rows: 940988


rows with an Infinity value across all 25 ratio columns: 0
path_match_score rows out of [0,1]: 0


rows where a NULL frechet distance wrongly produced a non-null score: 0
hand-check trip 1: 1158 / 1545.5891959798994 = 0.749229  (stored: 0.749229)
all checks passed


## Querying: GTFS route shape + AVL positions for a trip

Both joins are PK lookups (confirmed ~5ms/~0ms via `EXPLAIN ANALYZE`),
so this stays fast regardless of table size - no need to touch
`silver` or scan anything:

```sql
SELECT
    d.trip_id, d.bus_id, d.route_id, d.avl_matched,
    rs.shape_geom AS gtfs_route_geom,   -- official route line (LineString)
    pos.metric_timestamp, pos.geom AS bus_position
FROM ml.trip_validity_dataset d
LEFT JOIN ml.trip_validity_route_shapes rs
    ON rs.feed_version_date = d.gtfs_feed_version_date
   AND rs.shape_id = COALESCE(d.gtfs_shape_id_i, d.gtfs_shape_id_v)
LEFT JOIN ml.trip_validity_trip_positions pos
    ON pos.trip_id = d.trip_id
WHERE d.trip_id = 12345
ORDER BY pos.metric_timestamp;
```

Both are `LEFT JOIN`s on purpose: `rs` can be empty (~30 routes never
matched any GTFS feed) and `pos` can be empty (`avl_matched = false`,
or matched but no pings actually fell in the trip's window) - check
`gtfs_feed_version_date`/`avl_matched` if you need to tell "no route" /
"no positions" apart from "not queried yet". `COALESCE(..._i, ..._v)`
picks a default when `gtfs_route_has_both_directions` is true; use
`path_match_score_frechet_i`/`_v` (already computed) to pick whichever
shape this trip's *own* GPS path actually matches better, instead.